In [0]:
import json
from pyspark.sql import Row
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Flattening Teams reference data

In [0]:
def extract_teams(teams_json: dict) -> list[dict]:
    rows = []
    for t in teams_json.get("teams", []):
        rows.append({
            "team_id": t.get("id"),
            "team_name": t.get("name"),
            "team_abbreviation": t.get("abbreviation"),
            "league_name": t.get("league", {}).get("name"),
            "division_name": t.get("division", {}).get("name"),
            "venue_name": t.get("venue", {}).get("name")
        })
    return rows

In [0]:
bronze_teams = spark.table("bronze.mlb_teams_raw").collect()

all_team_rows = []
for row in bronze_teams:
    try:
        parsed = json.loads(row.raw_json)
        all_team_rows.extend(extract_teams(parsed))
    except json.JSONDecodeError as e:
        print(f"[WARN] failed to parse teams JSON: {e}")
    

## Writing teams into Silver

In [0]:
silver_teams = spark.createDataFrame([Row(**r) for r in all_team_rows]) \
    .dropDuplicates(["team_id"]) \
    .withColumn("silver_processed_at", current_timestamp())

silver_teams.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.mlb_teams")

print(f"Silver.mlb_teams rows: {silver_teams.count()}")
display(silver_teams)